In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone 1 - NLP Foundation & Semantic Similarity

# Question 1

## Frequency Distribution of Correct Answers

Calculate the frequency distribution of the correct answers (A, B, C, D, E) in `train.csv`.

**Answer:** 814

In [2]:
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [3]:
counts = df['answer'].value_counts()

most_freq = counts.iloc[0]
least_freq = counts.iloc[-1]

print(f"Most frequent count: {most_freq}")
print(f"Least frequent count: {least_freq}")
print(f"Sum: {most_freq + least_freq}")

# Question 2

## Vocabulary Size of Cleaned Prompts

After converting the `prompt` column to lowercase and removing all standard punctuation characters (`string.punctuation`), split the text by whitespace.

Determine the total number of unique words across the entire cleaned prompt column.

**Answer:** 859

In [4]:
import string

def clean_text(text):
    text = str(text).lower()
    # Remove punctuation using string.punctuation
    for p in string.punctuation:
        text = text.replace(p, '')
    # Split by whitespace
    return set(text.split())

global_vocab = set()
for prompt in df['prompt']:
    global_vocab.update(clean_text(prompt))

print(f"Total unique words: {len(global_vocab)}")

# Question 3

## Stop Word Removal

Using the cleaned prompt from Row ID 1, remove standard English stop words using:

`sklearn.feature_extraction.text.ENGLISH_STOP_WORDS`

Determine the number of words remaining.

**Answer:** 13

In [5]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Clean the first row's prompt
row_1_words = clean_text(df.iloc[0]['prompt'])

# Filter out stop words
filtered_words = [w for w in row_1_words if w not in ENGLISH_STOP_WORDS]

print(f"Words left after filtering: {len(filtered_words)}")

# Question 4

## TF-IDF Vocabulary Size

Fit a default TF-IDF vectorizer:

`TfidfVectorizer(stop_words='english')`

on the combined text of all prompts and options in `train.csv`.

Determine the total number of generated feature columns.

**Answer:** 2762

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

combined_text = (df['prompt'].fillna('') + ' ' + 
                 df['A'].fillna('') + ' ' + 
                 df['B'].fillna('') + ' ' + 
                 df['C'].fillna('') + ' ' + 
                 df['D'].fillna('') + ' ' + 
                 df['E'].fillna(''))

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text)

print(f"Total feature columns: {len(vectorizer.get_feature_names_out())}")

# Question 5

## Cosine Similarity for Row ID 1

Using the TF-IDF vectorizer from Question 4, calculate the cosine similarity between the prompt and Option A for Row ID 1.

Round the result to 4 decimal places.

**Answer:** 0.2720

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

# Transform the specific strings for Row 1
prompt_vec = vectorizer.transform([str(df.iloc[0]['prompt'])])
opt_a_vec = vectorizer.transform([str(df.iloc[0]['A'])])

sim_score = cosine_similarity(prompt_vec, opt_a_vec)[0][0]

print(f"Similarity Score: {round(sim_score, 4)}")

# Question 6

## Highest Similarity Accuracy

For every row in `train.csv`:

1. Calculate cosine similarity between the prompt and each option (A-E).
2. Select the option with the highest similarity.
3. Compare it with the correct answer.

Determine the percentage of correct predictions.

**Answer:** 13.55%

In [8]:
correct_matches = 0
options = ['A', 'B', 'C', 'D', 'E']

for idx, row in df.iterrows():
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    
    sims = [cosine_similarity(prompt_vec, vectorizer.transform([str(row[opt])]))[0][0] for opt in options]
    
    # Find the option with the highest score
    best_option = options[sims.index(max(sims))]
    
    if best_option == row['answer']:
        correct_matches += 1

percentage = (correct_matches / len(df)) * 100
print(f"Accuracy percentage: {round(percentage, 2)}%")

# Question 7

## MAP@3 Calculation (Case 1)

Ground Truth: C

Predictions:
C A B

**MAP@3 Score:** 1.0

# Question 8

## MAP@3 Calculation (Case 2)

Ground Truth: B

Predictions:
D B E

**MAP@3 Score:** 0.5

# Question 9

## Majority Class Baseline

Find the most frequent correct answer in the training set.

For every row:

- 1st prediction = Most frequent answer
- 2nd prediction = Second most frequent answer
- 3rd prediction = Third most frequent answer

Calculate the overall MAP@3 score.

**Answer:** 0.42125

In [9]:
import numpy as np

def apk(actual, predicted, k=3):
    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)

    if not actual:
        return 0.0
    return score / min(len(actual), k)

def mapk(actual_list, predicted_list, k=3):
    return np.mean([apk(a, p, k) for a, p in zip(actual_list, predicted_list)])

In [10]:
# Get the top 3 most frequent answers overall
top_3_classes = df['answer'].value_counts().index[:3].tolist()

actual_answers = [[ans] for ans in df['answer']]

majority_predictions = [top_3_classes for _ in range(len(df))]

majority_map3 = mapk(actual_answers, majority_predictions, k=3)
print(f"Majority Class Baseline MAP@3: {majority_map3}")

# Question 10

## TF-IDF Similarity Pipeline

For each row:

1. Compute TF-IDF cosine similarity between the prompt and all five options.
2. Rank options from highest to lowest similarity.
3. Select the top 3 predictions.
4. Calculate MAP@3.

Determine the average MAP@3 score across the entire training dataset.

**Answer:** 0.2961666666666667

In [11]:
tfidf_predictions = []

for idx, row in df.iterrows():
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    
    sim_dict = {opt: cosine_similarity(prompt_vec, vectorizer.transform([str(row[opt])]))[0][0] for opt in options}
    
    sorted_options = sorted(sim_dict.items(), key=lambda item: item[1], reverse=True)
    
    top_3_preds = [opt[0] for opt in sorted_options[:3]]
    tfidf_predictions.append(top_3_preds)

tfidf_map3 = mapk(actual_answers, tfidf_predictions, k=3)
print(f"TF-IDF Pipeline MAP@3: {tfidf_map3}")